# Practical 5 — Polynomial Regression

**Aim:** Study how polynomial degree affects model fit and generalization using the California Housing dataset.

## 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## 2. Load the California Housing dataset

Use median income (MedInc) as the input feature and median house value (MedHouseVal) as the target.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.sample(n=4000, random_state=42)

X = df[["MedInc"]]
y = df["MedHouseVal"]

print("Dataset shape:", df.shape)
print("Input feature: Median income")
print("Target: Median house value")
df[["MedInc", "MedHouseVal"]].head()

## 3. Split the dataset into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 4. Train polynomial regression models

Compare degrees 1, 2, 3, 5 and 8. Scaling is included in the pipeline to keep polynomial features on comparable scales.

In [ ]:
degrees = [1, 2, 3, 5, 8]
models = {}
results = []

for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    models[degree] = model
    results.append({
        "Degree": degree,
        "Train MSE": mean_squared_error(y_train, train_pred),
        "Test MSE": mean_squared_error(y_test, test_pred),
        "Test R2": r2_score(y_test, test_pred)
    })

## 5. Compare model performance

Lower MSE is better. A large gap between training and testing error can indicate overfitting.

In [ ]:
results_df = pd.DataFrame(results).round(4)
display(results_df)

best_degree = int(results_df.loc[results_df["Test MSE"].idxmin(), "Degree"])
print(f"Best degree based on test MSE: {best_degree}")

## 6. Plot the fitted polynomial curves

The graph shows how each degree captures the relationship between median income and median house value.

In [ ]:
x_line = pd.DataFrame({
    "MedInc": np.linspace(X["MedInc"].min(), X["MedInc"].max(), 300)
})

plt.figure(figsize=(10, 6))
plt.scatter(X_test["MedInc"], y_test, alpha=0.25, color="gray", label="Test data")

for degree, model in models.items():
    plt.plot(x_line["MedInc"], model.predict(x_line), linewidth=2, label=f"Degree {degree}")

plt.title("Polynomial Regression — Different Degrees")
plt.xlabel("Median Income")
plt.ylabel("Median House Value (in $100,000s)")
plt.ylim(0, 5.5)
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 7. Visualize the bias–variance trade-off

Compare training and testing MSE for each degree. Increasing degree may reduce training error while testing error can rise when the model starts to overfit.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["Degree"], results_df["Train MSE"], marker="o", linewidth=2, label="Training MSE")
plt.plot(results_df["Degree"], results_df["Test MSE"], marker="s", linewidth=2, label="Testing MSE")
plt.title("Model Complexity vs. Prediction Error")
plt.xlabel("Polynomial Degree")
plt.ylabel("Mean Squared Error")
plt.xticks(degrees)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Practical summary

In [ ]:
print("Polynomial Regression models a non-linear relationship by adding powers of the input feature. In this practical, median income is used to predict median house value in the California Housing dataset. Models with polynomial degrees 1, 2, 3, 5 and 8 are trained and compared using training MSE, testing MSE and R² score. The fitted curves and error plot show how model complexity affects prediction: a low degree may underfit, while an unnecessarily high degree may overfit. The most suitable degree is selected by comparing test-set performance, demonstrating the bias–variance trade-off.")